In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data  import DataLoader, TensorDataset

from  sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [87]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test =  train_test_split(X, y,  test_size  =0.2)

scaler =  StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [88]:
X_train_scaled

array([[-0.99846762,  1.37088983, -0.99557212, ..., -0.80899602,
         0.80887226, -0.13204936],
       [-0.42164078, -0.25780948, -0.3825879 , ...,  0.40141634,
         0.52615939,  1.08181073],
       [ 1.26269362, -0.40797325,  1.2296274 , ...,  0.79197037,
        -0.6110097 ,  0.58970529],
       ...,
       [ 1.75011231,  1.08211336,  2.16497116, ...,  2.08110378,
         4.01663674,  0.87829594],
       [-0.27454993,  0.28971071, -0.32538161, ..., -1.1051153 ,
        -0.33145563, -0.53129423],
       [ 0.24747837,  0.10951419,  0.15231181, ..., -0.867274  ,
        -0.39621109, -1.27862724]], shape=(455, 30))

In [89]:
y_train

array([1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0,
       1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1,
       0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0,
       0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1,

In [90]:
X_train_scaled_tensor =  torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor =  torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)


In [91]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)


In [92]:
X_train_scaled_tensor.shape

torch.Size([455, 30])

In [93]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [109]:
class BCNet(nn.Module):

    def  __init__(self):
        super(BCNet, self).__init__()

        self.fcl = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fcl(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x))

        return x
    

In [110]:
model =  BCNet()

In [111]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)


In [112]:
epochs = 20

for epoch  in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        preds = model(x_batch)

        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        running_loss  += loss.item()


    print(f'Epoch {epoch+1}: Loss was {running_loss / len(train_loader)}')

Epoch 1: Loss was 0.6603249271710714
Epoch 2: Loss was 0.5355173846085867
Epoch 3: Loss was 0.3800002952416738
Epoch 4: Loss was 0.23927682042121887
Epoch 5: Loss was 0.17408724625905356
Epoch 6: Loss was 0.13563445558150608
Epoch 7: Loss was 0.10635370214780172
Epoch 8: Loss was 0.09468324979146321
Epoch 9: Loss was 0.07106485422700644
Epoch 10: Loss was 0.0718944493525972
Epoch 11: Loss was 0.06219542995095253
Epoch 12: Loss was 0.05391637571156025
Epoch 13: Loss was 0.062069745485981306
Epoch 14: Loss was 0.05282861997062961
Epoch 15: Loss was 0.03922598975089689
Epoch 16: Loss was 0.03914293715109428
Epoch 17: Loss was 0.05069579565897584
Epoch 18: Loss was 0.044517241915067035
Epoch 19: Loss was 0.04187095674375693
Epoch 20: Loss was 0.0384921470656991


In [113]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >=0.5)  == y_test_tensor).float().mean().item()

In [108]:
accuracy

0.9649122953414917